# Dataset Exploration and Processing

In [4]:
from pathlib import Path
import json
import re
import subprocess
import sys

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
ROOT = next(
    path for path in (cwd, *cwd.parents)
    if (path / "scripts" / "prepare_data.py").is_file()
)
sys.path.insert(0, str(ROOT))

from scripts.prepare_data import (
    PreparationConfig,
    clean_speech_text,
    prepare_records,
    write_outputs,
)

DOWNLOAD_DIR = ROOT / "data" / "download"
RAW_OUTPUT = ROOT / "data" / "raw" / "speeches.csv"
PROCESSED_OUTPUT = ROOT / "data" / "processed" / "training_data.jsonl"
MANIFEST_OUTPUT = ROOT / "data" / "processed" / "manifest.json"

print("Repository:", ROOT)
print("Download-Verzeichnis:", DOWNLOAD_DIR)
print("Vorhanden:", DOWNLOAD_DIR.exists())

Repository: /home/marleen/merz-gpt
Download-Verzeichnis: /home/marleen/merz-gpt/data/download
Vorhanden: True


## 01 - Download Data

In [15]:
exploration_periods = (14, 15, 16, 20, 21)
download_command = [
    sys.executable,
    str(ROOT / "scripts" / "prepare_data.py"),
    "--download",
    "--periods",
    *map(str, exploration_periods),
    "--first-name",
    "Friedrich",
    "--last-name",
    "Merz",
]

RUN_DOWNLOAD = False
if RUN_DOWNLOAD:
    subprocess.run(download_command, check=True)


## 02 - Exploration of Exports

In [16]:
speech_files = sorted(DOWNLOAD_DIR.glob("bundestag_wp*_speeches.csv"))
person_files = sorted(DOWNLOAD_DIR.glob("bundestag_wp*_persons.csv"))

print("Gefundene Speech-Dateien:")
for path in speech_files:
    print(" -", path.name)

print("\nGefundene Person-Dateien:")
for path in person_files:
    print(" -", path.name)

if not speech_files or not person_files:
    raise FileNotFoundError("Noch keine Exporte gefunden. Zuerst den Download ausführen.")

Gefundene Speech-Dateien:
 - bundestag_wp14_speeches.csv
 - bundestag_wp15_speeches.csv
 - bundestag_wp16_speeches.csv
 - bundestag_wp20_speeches.csv
 - bundestag_wp21_speeches.csv

Gefundene Person-Dateien:
 - bundestag_wp14_persons.csv
 - bundestag_wp15_persons.csv
 - bundestag_wp16_persons.csv
 - bundestag_wp20_persons.csv
 - bundestag_wp21_persons.csv


In [7]:
def period_from_filename(path: Path) -> int:
    match = re.search(r"wp(\d+)", path.name)
    if match is None:
        raise ValueError(f"Keine Wahlperiode im Dateinamen: {path.name}")
    return int(match.group(1))


person_frames = [
    pd.read_csv(path).assign(source_period=period_from_filename(path))
    for path in person_files
]
persons = pd.concat(person_frames, ignore_index=True).drop_duplicates()

print(f"Personenzeilen nach Deduplizierung: {len(persons):,}")
display(persons.head(3))

Personenzeilen nach Deduplizierung: 6,025


,id,first_name,last_name,name_suffix,title,party,role,function,ministry,state,source_period
0,116824,Unknown,Unknown,NaN,Person 116824,NaN,NaN,NaN,NaN,NaN,14
1,115982,Unknown,Unknown,NaN,Person 115982,NaN,NaN,NaN,NaN,NaN,14
2,116120,Unknown,Unknown,NaN,Person 116120,NaN,NaN,NaN,NaN,NaN,14


In [8]:
print("Verfügbare Personenspalten:")
print(persons.columns.tolist())

name_columns = [column for column in persons.columns if "name" in column.casefold()]
print("\nNamensspalten:", name_columns)

merz_mask = persons.apply(
    lambda row: row.astype(str).str.contains("Merz", case=False, na=False).any(),
    axis=1,
)
merz_mentions = persons.loc[merz_mask]
print(f"Personenzeilen mit 'Merz' an irgendeiner Stelle: {len(merz_mentions)}")
display(
    merz_mentions[["id", "first_name", "last_name", "source_period"]]
    .drop_duplicates()
    .sort_values(["last_name", "first_name", "source_period"])
)

Verfügbare Personenspalten:
['id', 'first_name', 'last_name', 'name_suffix', 'title', 'party', 'role', 'function', 'ministry', 'state', 'source_period']

Namensspalten: ['first_name', 'last_name', 'name_suffix']
Personenzeilen mit 'Merz' an irgendeiner Stelle: 2


,id,first_name,last_name,source_period
3960,11002735,Friedrich,Merz,20
5264,11002735,Friedrich,Merz,21


In [9]:
exact_name_match = (
    persons["first_name"].fillna("").str.strip().str.casefold().eq("friedrich")
    & persons["last_name"].fillna("").str.strip().str.casefold().eq("merz")
)
merz_persons = persons.loc[exact_name_match].copy()
merz_ids = set(
    merz_persons["id"].dropna().astype(str).str.replace(r"\.0$", "", regex=True)
)

print("Exakt zugeordnete Speaker-IDs:", sorted(merz_ids))
display(merz_persons[["id", "first_name", "last_name", "source_period"]].drop_duplicates())

Exakt zugeordnete Speaker-IDs: ['11002735']


,id,first_name,last_name,source_period
3960,11002735,Friedrich,Merz,20
5264,11002735,Friedrich,Merz,21


In [10]:
speech_frames = [
    pd.read_csv(path, low_memory=False).assign(source_period=period_from_filename(path))
    for path in speech_files
]
speeches = pd.concat(speech_frames, ignore_index=True)

print(f"Reden insgesamt: {len(speeches):,}")
display(speeches.groupby("source_period").size().rename("speeches").to_frame())
display(speeches.head(3))

Reden insgesamt: 70,858


,speeches
source_period,
14,11968
15,8564
16,12412
20,26138
21,11776


,id,title,text,date,protocol_id,protocol_number,page_start,page_end,topics,speaker_id,...,is_presidential_announcement,extraction_method,extraction_status,extraction_confidence,is_xml_extracted,is_complete,is_high_confidence,is_president,page_section,source_period
0,667203,"Klaus Wowereit, Bundesratspräs.",[EXTRACTION_FAILED:ALL_STRATEGIES_FAILED] Spee...,2002-09-27,1087,780,445A,NaN,NaN,116824,...,False,none,failed,0.0,False,False,False,False,NaN,14
1,666879,"Gudrun Schaich-Walch, Parl. Staatssekr., Bunde...",[EXTRACTION_FAILED:ALL_STRATEGIES_FAILED] Spee...,2002-09-27,1087,780,456B-D,NaN,Pflegeheim,115982,...,False,none,failed,0.0,False,False,False,False,NaN,14
2,665289,"Margareta Wolf, Parl. Staatssekr., Bundesminis...",[EXTRACTION_FAILED:ALL_STRATEGIES_FAILED] Spee...,2002-09-27,1087,780,449D-450B,NaN,Korruptionsregister,116120,...,False,none,failed,0.0,False,False,False,False,NaN,14


In [11]:
speech_ids = speeches["speaker_id"].astype(str).str.replace(r"\.0$", "", regex=True)
merz_candidates = speeches.loc[speech_ids.isin(merz_ids)].copy()

print(f"Reden nach Speaker-ID-Filter: {len(merz_candidates):,}")
print("\nExtraktionsqualität:")
display(
    pd.crosstab(
        merz_candidates["extraction_method"],
        merz_candidates["extraction_status"],
        margins=True,
    )
)

merz_speeches = merz_candidates.loc[
    merz_candidates["extraction_method"].eq("xml")
    & merz_candidates["extraction_status"].eq("complete")
    & merz_candidates["text"].notna()
].copy()
print(f"Vollständige XML-Reden: {len(merz_speeches):,}")

Reden nach Speaker-ID-Filter: 207

Extraktionsqualität:


extraction_status,complete,All
extraction_method,,
xml,207,207
All,207,207


Vollständige XML-Reden: 207


## 03 - Cleaning Speech Texts

The XML texts contain stage directions such as applause, interjections, or laughter.

In [12]:
comparison = merz_speeches[["date", "protocol_id", "text"]].head(5).copy()
comparison["clean_text"] = comparison["text"].map(clean_speech_text)
comparison["raw_chars"] = comparison["text"].str.len()
comparison["clean_chars"] = comparison["clean_text"].str.len()
comparison["removed_chars"] = comparison["raw_chars"] - comparison["clean_chars"]

display(comparison[["date", "protocol_id", "raw_chars", "clean_chars", "removed_chars"]])

example = comparison.iloc[0]
print("Original (gekürzt):\n", example["text"][:280])
print("\nBereinigt (gekürzt):\n", example["clean_text"][:280])

,date,protocol_id,raw_chars,clean_chars,removed_chars
32978,2025-03-18,5701,14989,13179,1810
33023,2025-03-13,5700,20154,17611,2543
33101,2025-02-11,5698,30929,24277,6652
33147,2025-01-31,5697,15599,11799,3800
33149,2025-01-31,5697,1533,822,711


Original (gekürzt):
 Frau Präsidentin! Liebe Kolleginnen und Kollegen!

(Stephan Brandner [AfD]: Lieber Pinocchio-Fritze!)

Wir wollen heute eine sehr weitreichende – –

Entschuldigung. – Herr Brandner, dafür kriegen Sie jetzt einen Ordnungsruf.

(Beifall bei Abgeordneten der SPD und der CDU/CSU – Zu

Bereinigt (gekürzt):
 Frau Präsidentin! Liebe Kolleginnen und Kollegen!

Wir wollen heute eine sehr weitreichende – –

Entschuldigung. – Herr Brandner, dafür kriegen Sie jetzt einen Ordnungsruf.

Wir wollen heute eine sehr weitreichende, von vielen Menschen in unserem Land auch mit erheblichen Sorgen 


## 04 - Generating the Final Corpus

In [17]:
final_periods = (20, 21)
config = PreparationConfig(
    download_dir=DOWNLOAD_DIR,
    raw_output=RAW_OUTPUT,
    processed_output=PROCESSED_OUTPUT,
    manifest_output=MANIFEST_OUTPUT,
    periods=final_periods,
    first_name="Friedrich",
    last_name="Merz",
)

prepared_speeches, training_records = prepare_records(config)
print(f"Vorbereitet: {len(training_records)} eindeutige Reden aus WP {final_periods}")

EXPORT = False
if EXPORT:
    write_outputs(config, prepared_speeches, training_records)
    print("Geschrieben nach:", PROCESSED_OUTPUT)


Vorbereitet: 207 eindeutige Reden aus WP (20, 21)


In [14]:
total_chars = sum(len(record["text"]) for record in training_records)
summary = pd.Series(
    {
        "Reden nach Qualitätsfilter": len(prepared_speeches),
        "Wahlperioden": sorted(prepared_speeches["legislative_period"].unique().tolist()),
        "Frühestes Datum": prepared_speeches["date"].min(),
        "Spätestes Datum": prepared_speeches["date"].max(),
        "Zeichen gesamt": total_chars,
        "Mittlere Zeichen pro Rede": round(total_chars / len(training_records)),
    },
    name="Trainingskorpus",
)
display(summary.to_frame())

print("\nErster JSONL-Datensatz (gekürzt):")
print(json.dumps(training_records[0], ensure_ascii=False)[:500] + " ...")

,Trainingskorpus
Reden nach Qualitätsfilter,207
Wahlperioden,"[20, 21]"
Frühestes Datum,2022-01-27
Spätestes Datum,2026-07-09
Zeichen gesamt,996074
Mittlere Zeichen pro Rede,4812



Erster JSONL-Datensatz (gekürzt):
{"text": "Frau Präsidentin! Meine sehr geehrten Damen und Herren! Wir stehen vermutlich alle noch unter dem Eindruck der Reden, die wir heute Morgen zum Jahrestag der Befreiung des Konzentrationslagers Auschwitz gehört haben. Es erfüllt mich mit etwas Beklemmung, wenige Minuten später hier im Deutschen Bundestag über die Frage reden zu müssen, ob nicht möglicherweise erneut ein Krieg in Europa droht – ein Krieg, Frau Baerbock, kein Fußballspiel.\n\nHerr Bundeskanzler, wir hätten uns durchaus vor ...


## Final Reproducible Run
For a full run from the repository root, use:

```bash
python scripts/prepare_data.py --download --periods 20 21 --first-name Friedrich --last-name Merz
```
